# 330 · Find the qubit: widening the search window — solution

**Level:** advanced · **Topics:** automation, workflow, applications-library, spectroscopy, sweeps · **Type:** write the code

Solution of [exercise.ipynb](exercise.ipynb).

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np

from laboneq import serializers
from laboneq.automation import AutomationStatus
from laboneq.dsl.result.acquired_result import AcquiredResult
from laboneq.dsl.result.results import Results
from laboneq.simple import *
from laboneq_applications.automation import WorkflowAutomation, WorkflowLayer
from laboneq_applications.automation.workflow.logic import WorkflowLogic
from laboneq_applications.experiments import qubit_spectroscopy
from laboneq_applications.qpu_types.tunable_transmon.demo_qpus import demo_platform

## Device setup, QPU and session

A ready-made demo setup with three qubits.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html) ·
[Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
platform = demo_platform(n_qubits=3)
qpu = platform.qpu
qubits = ["q0", "q1", "q2"]

session = Session(platform.setup, log_level=logging.WARNING)
session.connect(do_emulation=True)

for qubit in qubits:
    print(qubit, "expected at", qpu[qubit].parameters.resonance_frequency_ge / 1e9, "GHz")

## The fake measurement

This part is given, there is nothing to change.

- `PEAKS` holds the true position and width of each qubit peak. Your code must not use it.
- `FakeSpectroscopyLayer` is a normal `WorkflowLayer` that first writes a fake spectrum for the current frequency
  windows to a file. The `inject_results` option of the workflow then replaces the emulated data with that file.
- `prepare` does nothing here. It is used in the stretch goal.

📖 [Automation](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/07a_automation/tutorials/00_automation.html) ·
[Experiment workflow automation](https://docs.zhinst.com/labone_q_user_manual/applications_library/tutorials/sources/experiment_workflow_automation.html)

In [ ]:
PEAKS = {"q0": (6.515e9, 3e6), "q1": (6.450e9, 12e6), "q2": (6.630e9, 6e6)}
RESULTS_FILE = "fake_results.json"
random = np.random.default_rng(seed=1)


def fake_signal(qubit, frequencies):
    center, width = PEAKS[qubit]
    peak = 1 / (1 + ((frequencies - center) / (width / 2)) ** 2)
    noise = 0.03 * random.standard_normal(len(frequencies))
    return (0.2 + peak + noise) * np.exp(0.3j)


class FakeSpectroscopyLayer(WorkflowLayer):
    def prepare(self, auto):
        pass

    def run_executable_core(self, auto):
        self.prepare(auto)
        data = {}
        for qubit, parameters in self.element_workflow_parameters.items():
            frequencies = parameters["frequencies"]
            data[f"{qubit}/result"] = AcquiredResult(
                data=fake_signal(qubit, frequencies),
                axis_name=[qubit],
                axis=[frequencies],
            )
        serializers.save(Results(acquired_results=data), RESULTS_FILE)
        return super().run_executable_core(auto)

## Parameters of the automation

One entry per layer. Every qubit starts with a window of ±10 MHz around its expected frequency.
A qubit **passes** when the fit of the peak is good (R² above 0.8). Then its frequency is updated in the QPU.

📖 [Qubit spectroscopy workflow](https://docs.zhinst.com/labone_q_user_manual/applications_library/how-to-guides/sources/01_superconducting_qubits/01_workflows/03_qubit_spectroscopy.html)

In [ ]:
options = {
    "count": 16,
    "evaluate": True,
    "update": True,
    "close_figures": True,
    "save_figures": False,
    "inject_results": {"paths": [RESULTS_FILE], "use": "emulation", "check_shapes": False},
}
evaluation = {
    "fit_r2_thresholds": {qubit: 0.8 for qubit in qubits},
    "parameter_thresholds": {qubit: 0 for qubit in qubits},
}

start_windows = {}
for qubit in qubits:
    expected = qpu[qubit].parameters.resonance_frequency_ge
    start_windows[qubit] = {"frequencies": np.linspace(expected - 10e6, expected + 10e6, 201)}

parameters = {
    "coarse": {
        "workflow_parameters": start_windows,
        "options": options,
        "evaluation_parameters": evaluation,
    },
    "fine": {
        "workflow_parameters": {qubit: {"frequencies": None} for qubit in qubits},
        "options": options,
        "evaluation_parameters": evaluation,
    },
}

## Step 1: the logic

After every run of a layer, its **logic** decides what happens next. It returns two things:

- the key of the layer to run next (the same layer, to try again),
- the parameters to change, in the same structure as `parameters["coarse"]`.

Write a logic that, for every qubit that failed, keeps the centre of the window and doubles its width.

Useful: `layer.target_nodes` (qubit → node), `node.status`, `AutomationStatus.FAILED`,
`layer.workflow_parameters[qubit]["frequencies"]`.

In [ ]:
class WidenWindow(WorkflowLogic):
    def run_executable_core(self, layer):
        new_parameters = {}
        for qubit, node in layer.target_nodes.items():
            if node.status == AutomationStatus.FAILED:
                frequencies = layer.workflow_parameters[qubit]["frequencies"]
                center = (frequencies[0] + frequencies[-1]) / 2
                width = 2 * (frequencies[-1] - frequencies[0])
                new_parameters[qubit] = {
                    "frequencies": np.linspace(
                        center - width / 2, center + width / 2, len(frequencies)
                    )
                }
        return layer.key, {"workflow_parameters": new_parameters}

## Step 2: run the search

A node may fail 4 times by default, then the automation gives up on it. `q2` needs a fifth attempt,
so `max_fail_count` is raised.

In [ ]:
auto = WorkflowAutomation(session, qpu, name="peak_search", parameters=parameters)

coarse = FakeSpectroscopyLayer(
    qubit_spectroscopy.experiment_workflow, qubits, key="coarse", depends_on={"root"}
)
coarse.logic = WidenWindow()
coarse.max_fail_count = {qubit: 8 for qubit in qubits}
auto.add_layer(coarse)

auto.run()

`report` prints the outcome of a layer and `plot_layer` shows the last spectrum of each qubit with its fit. Both are given.

In [ ]:
def report(layer_key):
    layer = auto[layer_key]
    for qubit in qubits:
        frequencies = layer.parameters["workflow_parameters"][qubit]["frequencies"]
        found = qpu[qubit].parameters.resonance_frequency_ge
        error = abs(found - PEAKS[qubit][0])
        print(
            f"{qubit}: {layer[qubit].status}, {layer.fail_count[qubit]} failed attempts, "
            f"window {(frequencies[-1] - frequencies[0]) / 1e6:.0f} MHz, "
            f"error {error / 1e3:.0f} kHz"
        )

In [ ]:
def plot_layer(layer_key):
    layer = auto[layer_key]
    if not layer.workflow_results:
        print(f"The layer '{layer_key}' has no results to plot.")
        return
    figure, axes = plt.subplots(1, len(qubits), figsize=(12, 3))
    for axis, qubit in zip(axes, qubits):
        for qubit_group, workflow_result in layer.workflow_results.items():
            if qubit not in qubit_group:
                continue
            fits = workflow_result.tasks["analysis_workflow"].tasks["fit_data"].output
            if qubit not in fits:
                continue
            fit = fits[qubit]
            frequencies = fit.userkws["x"] / 1e9
            axis.plot(frequencies, fit.data, ".", label="data")
            axis.plot(frequencies, fit.best_fit, label="fit")
        axis.set_title(f"{layer_key} · {qubit} · {layer[qubit].status}")
        axis.set_xlabel("frequency (GHz)")
        axis.legend()
    axes[0].set_ylabel("signal magnitude")
    figure.tight_layout()
    plt.show()

In [ ]:
# check
report("coarse")

**Expected result:** all three qubits `passed`, with windows of 40, 160 and 320 MHz and errors below 100 kHz.

```
q0: passed, 1 failed attempts, window 40 MHz, ...
q1: passed, 3 failed attempts, window 160 MHz, ...
q2: passed, 4 failed attempts, window 320 MHz, ...
```

The last attempt of each qubit, with the fit. A qubit that passed shows a clear peak with the fit on top of it.

In [ ]:
plot_layer("coarse")

## Stretch goal: a fine scan

The wide windows found the peaks, but with a coarse frequency step. Add a second layer that runs after `coarse`
and scans ±10 MHz around the frequency that was just found.

The coarse layer has written the new frequency to the QPU: `auto.qpu[qubit].parameters.resonance_frequency_ge`.
Use `prepare` to set the window of each qubit just before the layer runs.

In [ ]:
class FineScanLayer(FakeSpectroscopyLayer):
    def prepare(self, auto):
        for qubit in self.quantum_elements:
            center = auto.qpu[qubit].parameters.resonance_frequency_ge
            self.workflow_parameters[qubit]["frequencies"] = np.linspace(
                center - 10e6, center + 10e6, 201
            )

In [ ]:
fine = FineScanLayer(
    qubit_spectroscopy.experiment_workflow, qubits, key="fine", depends_on={"coarse"}
)
auto.add_layer(fine)

auto.run(from_layer="fine")

In [ ]:
# check
report("fine")

**Expected result:** all three qubits `passed` with a 20 MHz window and no failed attempt. The errors of `q1` and `q2` are smaller than after the coarse search.

The fine scan: each peak now fills its window.

In [ ]:
plot_layer("fine")

## Answers

1. The logic only changes the parameters of the qubits that failed. All qubits of the layer run again at every attempt, but a qubit that passed keeps its window.
2. The frequency step doubles as well. Once the step is close to the width of the peak, only one or two points sit on it, the fit gets poor and the peak can be missed. A narrow peak far away is the worst case. Increasing the number of points with the window avoids it, at the price of longer measurements.
3. The window keeps doubling until `max_fail_count` is reached, then the qubit stays `failed`. On an instrument the window is also limited by the hardware: far enough from the local oscillator the compilation fails. A real logic should stop at a maximum width.
4. Its node for that qubit is deactivated, because it depends on the failed node of `coarse`. The other qubits continue.

**Takeaway:** a layer runs an experiment on every qubit, and its logic turns the failures into new parameters for the next attempt. Layers that depend on it only run for the qubits that passed.